In [ ]:
# 5 每组30个cell PD merge图展示
import matplotlib.cm as cm
import numpy as np
import matplotlib.pyplot as plt
import pandas as pda
import homcloud.interface as hc
from matplotlib.lines import Line2D

save_path1 = pd_dir  

def pdIMAGE_group(data, i, region):

    # 每次独立实验一种颜色
    experiments = sorted(data['Experiment_ID'].unique())
    exp_colors = cm.tab10(np.linspace(0, 1, len(experiments)))

    color_dict = dict(zip(experiments, exp_colors))

    plt.figure(figsize=(5,5))

    for label in data['label1'].unique():

        samplename = label

        # 获取当前细胞所属实验
        experiment = data.loc[
            data['label1'] == label, 'Experiment_ID'
        ].iloc[0]

        color = color_dict[experiment]

        # Load persistence diagram
        pds = hc.PDList(
            save_path1 + samplename + "grayscale" + region + ".pdgm"
        )

        pd = pds.dth_diagram(i)

        births_raw = np.array(pd.births)
        deaths_raw = np.array(pd.deaths)

        # 保留原始 birth-death 坐标，不再使用 birth >= -1 过滤
        coords = pda.DataFrame({
            "birth": births_raw,
            "death": deaths_raw
        })

        # 统计相同坐标的频数
        freq = coords.value_counts().reset_index(name="count")

        # count 用来控制点大小
        plt.scatter(
            freq["birth"],
            freq["death"],
            s=np.sqrt(freq["count"]) * 0.2,#1.8 flag
            color=color,
            alpha=0.6
        )

    plt.axline((0, 0), slope=1, color='gray')

    plt.xlim(minv, maxv)
    plt.ylim(minv, maxv)

    plt.title('')
    plt.xlabel("Birth", fontsize=22)
    plt.ylabel("Death", fontsize=22)
    plt.tick_params(axis='both', which='major', labelsize=18)

  
    legend_elements = [
        Line2D(
            [0], [0],
            marker='o',
            color='w',
            label=experiment,
            markerfacecolor=color_dict[experiment],
            markersize=10
        )
        for experiment in experiments
    ]

    plt.legend(
        handles=legend_elements,
        loc='upper left',
        fontsize=12
    )

    print(region, '细胞数:', data['label1'].nunique())

    plt.savefig(
        path + f'd{i}_' + region + f'pd_merge_byexperiment_countsize_{tag}.png',
        dpi=300,
        bbox_inches='tight'
    )

    plt.show()


# Three groups
pdIMAGE_group(s, 0, 'SNCA')
pdIMAGE_group(u, 0, 'UBL3')
pdIMAGE_group(c, 0, 'CAAX')

In [ ]:
def betti_curve(i,df,region,samplename,save_path1,color,xs,plot=True):
    import numpy as np
    import matplotlib.pyplot as plt
    import pandas as pda
    import homcloud.interface as hc
    
    pds = hc.PDList(save_path1+samplename+"grayscale"+region+".pdgm")
    pd = pds.dth_diagram(i)

    births_raw = np.asarray(pd.births, dtype=float)
    deaths_raw = np.asarray(pd.deaths, dtype=float)
    finite = np.isfinite(births_raw) & np.isfinite(deaths_raw)
    births_cell = births_raw[finite]
    deaths_cell = deaths_raw[finite]


    if MODE == 'sublevel':
        ys = np.array([np.sum((births_raw <= x) & (deaths_raw > x)) for x in xs])
    else:  # superlevel：birth > death，阈值从高到低
        ys = np.array([np.sum((deaths_raw < x) & (x <= births_raw)) for x in xs])
    

    plt.plot(xs, ys,label=samplename,color = color)
    
    #plt.legend(fontsize=14)
    plt.xlabel('Threshold',fontsize=26)
    plt.ylabel('Number of connected components',fontsize=18)#connected components loops
    plt.tick_params(axis='both', which='major', labelsize=18)
    plt.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
    ax = plt.gca()
    y_offset = ax.yaxis.get_offset_text()
    y_offset.set_fontsize(18)

    #plt.title(f'{region}_d{i}')
    
    plt.savefig(path+f'{region}_d{i}_{tag}.png', bbox_inches='tight', pad_inches=0.3)
    return ys,births_cell, deaths_cell#births_cell[keep], deaths_cell[keep]
import matplotlib.cm as cm
import numpy as np
import pandas as pd

xs = np.linspace(minv, maxv, 140)

def betti_group(df, i, region):

    plt.figure(figsize=(5,5))
    colors = cm.tab20(np.linspace(0,1,len(df['label1'].unique())))

    YS = []
    experiment_ids = []
    cell_ids = []

    for idx, label in enumerate(df['label1'].unique()):

        ys, births, deaths = betti_curve(
            i,
            df,
            region,
            samplename=label,
            save_path1=pd_dir, 
            xs=xs,
            plot=True,
            color=colors[idx]
        )

        YS.append(ys)

        
        cell_info = df[df['label1'] == label].iloc[0]
        experiment_ids.append(cell_info['Experiment_ID'])
        cell_ids.append(label)

    YS = np.array(YS)

    print(region, '曲线数量:', len(YS))

    return YS, experiment_ids, cell_ids

In [ ]:
YS_s, EXP_s, CELL_s = betti_group(s, 0, 'SNCA')
YS_u, EXP_u, CELL_u = betti_group(u, 0, 'UBL3')
YS_c, EXP_c, CELL_c = betti_group(c, 0, 'CAAX')

In [ ]:
def betti_experiment_mean(YS, EXP, region):

    YS = np.asarray(YS)
    EXP = np.asarray(EXP)

    experiment_curves = {}

    for exp in sorted(np.unique(EXP)):

        mask = EXP == exp

        # 每次独立实验内，10个细胞取平均
        experiment_curves[exp] = np.mean(YS[mask], axis=0)

        print(region, exp, '细胞数:', np.sum(mask))

    return experiment_curves


BC_s = betti_experiment_mean(YS_s, EXP_s, 'SNCA')
BC_u = betti_experiment_mean(YS_u, EXP_u, 'UBL3')
BC_c = betti_experiment_mean(YS_c, EXP_c, 'CAAX')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# 每组分别计算三次独立实验平均曲线的总平均
y1 = np.mean(np.array(list(BC_s.values())), axis=0)
y2 = np.mean(np.array(list(BC_u.values())), axis=0)
y3 = np.mean(np.array(list(BC_c.values())), axis=0)

# 绘制三组平均 Betti curve
plt.figure(figsize=(5.6,5))

plt.plot(
    xs, y1, linewidth=2,
    label='Syn-HiBiT',
    color='#264a5f'
)

plt.plot(
    xs, y2, linewidth=2,
    label='Syn-HiBiT+AAV-Myc-UBL3',
    color='#d15034'
)

plt.plot(
    xs, y3, linewidth=2,
    label='Syn-HiBiT+AAV-Myc-UBL3∆5',
    color='#f1ae2d'
)

# Calculate AUC
AUCs = [
    np.trapezoid(y1, xs),
    np.trapezoid(y2, xs),
    np.trapezoid(y3, xs)
]

print('Mean Betti curve AUC:', AUCs)

# Legend: 保留原来的换行方式
handles, labels = plt.gca().get_legend_handles_labels()

new_handles = []
new_labels = []

for h, lab in zip(handles, labels):

    parts = lab.split('+')
    line1 = parts[0].strip()
    line2 = '+'.join(parts[1:]).strip() if len(parts) > 1 else ''

    new_handles.append(h)
    new_labels.append(line1)

    if line2 != '':
        empty = mpatches.Patch(facecolor='none', edgecolor='none')
        new_handles.append(empty)
        new_labels.append("  +" + line2)

plt.legend(
    new_handles, new_labels,
    fontsize=13,
    handlelength=1.5,
    labelspacing=0.2,
    borderpad=0.3,
    loc="upper right"
)

plt.xlabel('Threshold', fontsize=26)
plt.ylabel('Number of connected components', fontsize=18)

plt.tick_params(axis='both', which='major', labelsize=18)
plt.ticklabel_format(style='sci', axis='y', scilimits=(0,0))

ax = plt.gca()
ax.yaxis.get_offset_text().set_fontsize(18)

plt.savefig(
    path + f'd0_betti_curve_update_{tag}_0.1.png',
    bbox_inches='tight',
    pad_inches=0.3
)

plt.show()